# 03 – Xử lý, kiểm chứng và xuất sản phẩm trạm KAK (Bước 3–5)

**Bước 3:** Xử lý chuỗi thời gian địa từ trạm KAK theo các bước chuẩn trong SOP-01 (Nội suy mốc thời gian, xử lý Sentinel, gán mã `quality_flag` từ 0 đến 5).  
**Bước 4:** Kiểm chứng tính toàn vẹn bằng câu lệnh `assert` (Bảo toàn chính xác 1,707,840 dòng dữ liệu 1 phút).  
**Bước 5:** Xuất dữ liệu sạch Parquet, tự động cập nhật Data Dictionary và Quality Report cho KAK.


In [1]:
# ── Setup môi trường & nạp thư viện ──────────────────────────────────────────
import sys
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import config as cfg
from common.parsers import load_all_intermagnet
from kak.pipeline import run_pipeline, run_assertions
from common.quality_checks import run_all_checks
from common.export import export_parquet, write_data_dictionary, write_quality_report

print('Setup môi trường thành công.')


Setup môi trường thành công.


In [2]:
# ── Nạp kết quả QC trước xử lý (qc_before.pkl) ────────────────────────────────
qc_path = cfg.DATA_INTERIM / 'qc_before.pkl'
if qc_path.exists():
    with open(qc_path, 'rb') as f:
        qc_results_before = pickle.load(f)
    print(f'✅ Nạp thành công QC before: {list(qc_results_before.keys())}')
else:
    print('⚠  Không tìm thấy qc_before.pkl – Vui lòng chạy notebook 02_quality.ipynb trước')
    qc_results_before = {}


✅ Nạp thành công QC before: ['KAK']


In [3]:
# ── Bước 3: Chạy pipeline làm sạch dữ liệu KAK ───────────────────────────────
clean_dfs    = {}
proc_logs    = {}
station_meta = {}

for station in cfg.STATIONS:
    try:
        df_raw, meta = load_all_intermagnet(station)
        station_meta[station] = meta

        df_clean, log = run_pipeline(
            df_raw,
            station=station,
            sentinel_values=meta.get('sentinel_values', cfg.SENTINEL_VALUES)
        )
        clean_dfs[station] = df_clean
        proc_logs[station] = log
        print(f'✅ Trạm {station}: Làm sạch hoàn tất ({len(df_clean):,} dòng)')
    except FileNotFoundError as e:
        print(f'⚠  {e}')



  Pipeline | KAK
  Đầu vào: 1,707,840 dòng
  Bước 3.3: Thay 0 sentinel → NaN.
  Bước 3.4: Sau reindex: 1,707,840 dòng
  Bước 3.5: Phân phối cờ: {0: 1690346, 3: 15082, 5: 2412}
  Bước 3.6: 0 điểm được nội suy (cờ 1)
  Bước 3.7: Ép kiểu xong

  ✅ Pipeline hoàn tất | KAK: 1,707,840 dòng
✅ Trạm KAK: Làm sạch hoàn tất (1,707,840 dòng)


In [4]:
# ── Bước 4: Kiểm chứng tính toàn vẹn (Assertion checks) ─────────────────────
for station, df_clean in clean_dfs.items():
    qc_before = qc_results_before.get(station, {})
    run_assertions(
        df_clean,
        station=station,
        expected_rows=qc_before.get('gaps', {}).get('expected_points'),
        qc_before=qc_before
    )
    print(f'✅ Trạm {station}: Kiểm chứng assertion thành công!')



──────────────────────────────────────────────────
  Kiểm chứng (Bước 4) | KAK
──────────────────────────────────────────────────
  ✅ Số dòng: 1,707,840
  ✅ Timestamp duy nhất
  ✅ Timestamp tăng dần
  ✅ Không còn sentinel
  ✅ quality_flag trong phạm vi 0–5
  ✅ Khớp với kỳ vọng từ Bước 2

  🎉 Tất cả kiểm chứng ĐẠT | KAK
✅ Trạm KAK: Kiểm chứng assertion thành công!


In [5]:
# ── Bước 5a: Xuất dữ liệu sạch sang định dạng Parquet ────────────────────────
for station, df_clean in clean_dfs.items():
    export_parquet(
        df_clean,
        source='intermagnet',
        station=station,
        freq='1min',
        also_csv=False
    )


  💾 Parquet: /home/khang/Code/data-pipeline/geopulse-kak-usgs/data/clean/clean_intermagnet_KAK_1min_20230101_20260331.parquet (21,966,893 bytes)


In [6]:
# ── Bước 5b: Chạy QC sau xử lý (Để so sánh chỉ số Trước / Sau) ───────────────
qc_results_after = {}
for station, df_clean in clean_dfs.items():
    result_after = run_all_checks(df_clean, station=station, label='after')
    qc_results_after[station] = result_after
    print(f'✅ Trạm {station}: Hoàn thành QC sau xử lý')



──────────────────────────────────────────────────
  Kiểm tra chất lượng | KAK | AFTER
──────────────────────────────────────────────────
  1. Thiếu giá trị (NaN + sentinel)...
column  nan_count  sentinel_count  total_missing  missing_pct
  x_nt          0               0              0          0.0
  y_nt          0               0              0          0.0
  z_nt          0               0              0          0.0
  f_nt          0               0              0          0.0

  2. Trùng timestamp...
     Dòng liên quan: 0 | Trùng: 0

  3. Thiếu timestamp (gap)...
     Kỳ vọng: 1707840 | Thực tế: 1707840 | Thiếu: 0 | Số gap: 0 | Gap dài nhất: 0 phút

  4. Ngoài khoảng vật lý...
column  bound_min  bound_max  below_min  above_max  out_of_range  out_pct
  x_nt      20000      50000          0          0             0      0.0
  y_nt      -5000      10000          0          0             0      0.0
  z_nt      20000      60000          0          0             0      0.0
  f_nt    

In [7]:
# ── Bước 5c: Xuất Data Dictionary và Quality Report cho KAK ──────────────────
for station in clean_dfs:
    df_clean = clean_dfs[station]
    meta     = station_meta.get(station)

    print(f'\n── Trạm {station} ──')

    # Xuất Data Dictionary (Lấy schema thực tế từ DataFrame đã làm sạch)
    write_data_dictionary(df_clean, meta=meta)

    # Xuất Quality Report (So sánh chỉ số QC trước vs sau xử lý)
    write_quality_report(
        qc_before=qc_results_before.get(station, {}),
        qc_after=qc_results_after.get(station, {}),
        processing_log=proc_logs.get(station, {}),
        meta=meta
    )

print('\n✅ Tất cả sản phẩm KAK đã được xuất thành công!')
print(f'   Dữ liệu Parquet : {list((cfg.DATA_CLEAN).glob("*.parquet"))}')
print(f'   Báo cáo md      : {list(cfg.DOCS_DIR.glob("**/*.md"))}')



── Trạm KAK ──
  📄 Data dictionary: /home/khang/Code/data-pipeline/geopulse-kak-usgs/docs/kak/data_dictionary.md
  📄 Quality report: /home/khang/Code/data-pipeline/geopulse-kak-usgs/docs/kak/data_quality_report.md

✅ Tất cả sản phẩm KAK đã được xuất thành công!
   Dữ liệu Parquet : [PosixPath('/home/khang/Code/data-pipeline/geopulse-kak-usgs/data/clean/clean_usgs_JP_M4_20230101_20260331.parquet'), PosixPath('/home/khang/Code/data-pipeline/geopulse-kak-usgs/data/clean/clean_usgs_JP_M4_daily_20230101_20260331.parquet'), PosixPath('/home/khang/Code/data-pipeline/geopulse-kak-usgs/data/clean/clean_intermagnet_KAK_1min_20230101_20260331.parquet')]
   Báo cáo md      : [PosixPath('/home/khang/Code/data-pipeline/geopulse-kak-usgs/docs/scope.md'), PosixPath('/home/khang/Code/data-pipeline/geopulse-kak-usgs/docs/kak/data_quality_report.md'), PosixPath('/home/khang/Code/data-pipeline/geopulse-kak-usgs/docs/kak/data_dictionary.md'), PosixPath('/home/khang/Code/data-pipeline/geopulse-kak-usgs/doc

In [8]:
# ── Kiểm chứng khả năng tái lập (Reproducibility check) ───────────────────────
print('Kiểm chứng khả năng tái lập...')
for station in cfg.STATIONS:
    if station not in clean_dfs:
        continue
    df_raw2, meta2 = load_all_intermagnet(station)
    df_clean2, _ = run_pipeline(
        df_raw2, station=station,
        sentinel_values=meta2.get('sentinel_values', cfg.SENTINEL_VALUES)
    )
    # So sánh từng cột đo lường
    meas = [c for c in df_clean2.columns if c.endswith('_nt')]
    for col in meas:
        assert df_clean2[col].equals(clean_dfs[station][col]), f'❌ Không tái lập được: {station}/{col}'
    print(f'  ✅ Trạm {station}: Tái lập kết quả thành công 100%')

print('\n🎉 Pipeline xử lý KAK hoàn toàn tái lập được (Reproducible)!')


Kiểm chứng khả năng tái lập...

  Pipeline | KAK
  Đầu vào: 1,707,840 dòng
  Bước 3.3: Thay 0 sentinel → NaN.
  Bước 3.4: Sau reindex: 1,707,840 dòng
  Bước 3.5: Phân phối cờ: {0: 1690346, 3: 15082, 5: 2412}
  Bước 3.6: 0 điểm được nội suy (cờ 1)
  Bước 3.7: Ép kiểu xong

  ✅ Pipeline hoàn tất | KAK: 1,707,840 dòng
  ✅ Trạm KAK: Tái lập kết quả thành công 100%

🎉 Pipeline xử lý KAK hoàn toàn tái lập được (Reproducible)!
